# GPT mini tiếng Việt từ số 0 (portfolio)

Bạn pretrain một GPT tiếng Việt hoàn chỉnh: tokenizer riêng, lớp `GPT` của bài Basic 01, vòng train của bài Intermediate 01,
dữ liệu là toàn bộ bài Wikipedia dài của hai shard đã tải. Kết quả là một checkpoint tự chứa (trọng số, cấu hình, tokenizer)
mà module 08 (interpretability) mổ xẻ và module 10 (quantization) nén lại, cùng một báo cáo portfolio.

**Cách làm bài**
- Ba hàm được test chấm (cell dưới, đặc tả trong docstring). Phần còn lại tự thiết kế.
- GPU: chạy bằng kernel `ai-gpu` (xem `setup/gpu/README.md`); cấu hình laptop 4GB và PC 12GB ở README của bài.
- Chấm: `uv run pytest tracks/ai-engineer/modules/05-transformers-llm-architecture/exercises/advanced/01-gpt-mini-vietnamese -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
WIKI = ROOT / "data/raw/wikipedia-vi/20231101.vi"
print(sorted(p.name for p in WIKI.iterdir()))

In [ ]:
import dataclasses

import torch
from tokenizers import Tokenizer

## 1. Ba hàm được chấm

- `save_checkpoint(path, model, config, tokenizer, step, metrics)`: lưu **một** tệp bằng `torch.save` một dict có đúng các
  khoá `"format"` (chuỗi `"gpt-mini-v1"`), `"config"` (`dataclasses.asdict(config)`), `"state_dict"` (`model.state_dict()`,
  tensor đưa về CPU), `"tokenizer"` (`tokenizer.to_str()`), `"step"`, `"metrics"` (dict số). Module 08 và 10 đọc đúng định
  dạng này.
- `load_checkpoint(path, gpt_module, device="cpu")`: đọc tệp trên (`weights_only=True`), dựng
  `gpt_module.GPT(gpt_module.GPTConfig(**config))`, nạp trọng số, chuyển sang `device`, chế độ `eval`; trả
  `(model, tokenizer, info)` với `info = {"step": ..., "metrics": ...}`. Tệp có `"format"` khác thì `ValueError`.
- `distinct_n(ids, n)`: tỉ lệ n-gram **khác nhau** trên tổng số n-gram của một dãy token (đo văn bản sinh có lặp không);
  dãy ngắn hơn `n` trả `0.0`.

In [ ]:
def save_checkpoint(path, model: torch.nn.Module, config, tokenizer: Tokenizer, step: int, metrics: dict) -> None:
    # TODO
    raise NotImplementedError


def load_checkpoint(path, gpt_module, device: str = "cpu"):
    # TODO
    raise NotImplementedError


def distinct_n(ids: list[int], n: int) -> float:
    # TODO
    raise NotImplementedError

## 2. Gợi ý bố cục (không chấm)

1. **Dữ liệu**: validation là đúng 100 bài của module 04 (bài Intermediate 01 có hàm chia). Train là mọi bài dài còn lại của
   hai shard (đừng để bài validation lọt vào train). Ghi số bài, số ký tự, số token.
2. **Tokenizer**: BPE byte-level 8 000 giống module 04 để bpc so được; nếu muốn vocab khác, vẫn báo bpc.
3. **Model**: lớp `GPT` của bài Basic 01. Cấu hình laptop 4GB và PC 12GB ở README của bài.
4. **Train**: AdamW, warmup + cosine, clip 1.0, bf16 autocast trên GPU; đánh giá định kỳ bằng `evaluate_nll` của bài
   Intermediate 01 (cửa sổ trượt, stride bằng nửa context); lưu checkpoint tốt nhất bằng `save_checkpoint`.
5. **Đánh giá và sinh**: bpc trên validation so với LSTM (1.975) và GPT mini của bài Intermediate 01; mẫu sinh với vài
   temperature và top-k, đo `distinct_n`; nạp lại checkpoint bằng `load_checkpoint` và kiểm cho cùng bpc.
6. Viết `REPORT.md`.

In [ ]:
# explore: TODO: pipeline của bạn